# Ćwiczenia 2: Model ML, FastAPI i serwowanie predykcji

*Studia zaoczne | 1,5h*

## Cel

- Wytrenowanie modelu detekcji fraudów,
- Serwowanie modelu przez FastAPI,
- Podłączenie do Kafki — scoring w czasie rzeczywistym.

---

## Część 1: Trening modelu (25 min)

### Zadanie 1.1 — Przygotuj dane i wytrenuj model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

### Zadanie 1.2 — Podziel dane, wytrenuj, oceń

In [ ]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# TWÓJ KOD
# 1. train_test_split (80/20, stratify=y)
# 2. RandomForestClassifier(100)
# 3. classification_report
# 4. pickle.dump do 'fraud_model.pkl'

---

## Część 2: FastAPI (25 min)

### Zadanie 2.1 — Utwórz API serwujące model

In [ ]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

# TWÓJ KOD
# Endpoint POST /score
# Przyjmij Transaction, zwróć: {"is_fraud": bool, "fraud_probability": float}

Uruchom w terminalu:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

### Zadanie 2.2 — Przetestuj API

In [ ]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

# TWÓJ KOD — przetestuj podejrzaną transakcję (amount=5500, is_electronics=1, tx_per_minute=12)

---

## Część 3: Kafka + ML (25 min)

### Zadanie 3.1 — Konsument z scoringiem ML

Napisz konsumenta, który czyta z `transactions`, odpytuje API i wysyła alerty.

In [ ]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

# TWÓJ KOD
# Dla każdej transakcji:
# 1. Wyciągnij cechy (amount, hour z timestamp, is_electronics, tx_per_minute=5)
# 2. requests.post(API_URL, json=features)
# 3. Jeśli is_fraud: wyślij do tematu 'alerts', wypisz ALERT

### Zadanie 3.2 — Uruchom pipeline

W 3 terminalach:

1. `python producer.py` (z Ćw. 1)
2. `uvicorn fraud_api:app --host 0.0.0.0 --port 8001`
3. `python ml_consumer.py`

Obserwuj alerty.

---

## Praca domowa

1. Porównaj wyniki scoringu regułowego (Ćw. 1) vs ML — który lepiej wykrywa?
2. Dodaj endpoint `GET /health` do API.
3. Wypchnij do Git.

**Następne zajęcia:** Apache Spark — przetwarzanie na dużą skalę.